i need to do multiple things

- monitor how gradients look for each component at epochs
- look at bad cases and see what the gradients look like
- look at the stnadard deviation of w and codes
- look at the standard deviation of recons
- look at the correct recons loss (unscaled)



- For very eps, the gradient basically explodes. its of the scale 1e32, and recon loss does not change because of that.  

In [ ]:
def warmup_with_l2(
    model, epochs, n_samples, X_t, batch_size, sigma_eps, sigma_s, sigma_0, verbose=True
):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            _recon_loss = recon_loss(batch, recon, sigma_eps)
            _codes_loss = codes_loss(codes, sigma_s)
            _weights_loss = gauss_loss(model.decoder.weight, 0) / (sigma_0 * sigma_0)

            loss = _recon_loss + _codes_loss + _weights_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose and epoch % 200 == 0:
            print(f"warmup: epoch {epoch:4d} | recon_loss {_recon_loss:.4f}")


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import *
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
from pt_to_api import disjoint_ae, disjoint_ae_learned_sig
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment

In [ ]:
import warnings

MODE = "light"


def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]


def generate_synthetic_patches(
    patch_dim=72, n_components=10, k=3, n_samples=1000, noise_std=0.01, seed=42
):
    rng = np.random.RandomState(seed)

    dim_partition = make_dim_partition(patch_dim, n_components, seed)

    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)

    # each sample uses exactly k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        idx = rng.choice(n_components, k, replace=False)
        codes_true[i, idx] = rng.randn(k)

    X = codes_true @ W_true
    X += rng.randn(*X.shape) * noise_std

    return X, W_true, codes_true, dim_partition


def show_closest_component_of_W_for_each_component(components, W_true, figsize=(5, 2)):
    """
    Given two arrays of numpy vectors of same shapes
    for every component in `components`, this function shows the array in `W_true`
    which has the maximum cosine similarity with the component
    """
    sims = np.abs(cosine_similarity(components, W_true))
    pairs = []
    for i in range(len(components)):
        j = np.argmax(sims[i])
        pairs.append((i, j, sims[i][j]))
    for i, j, score in pairs:
        S(
            [components[i].reshape(3, 3), W_true[j].reshape(3, 3)],
            figsize,
            mode=MODE,
            suptitle=f"similarity score={score}",
            ax_titles=["component", "ground_truth"],
            viztype="local",
        )
        plt.show()


def evaluate_recovery(W_learned, W_true, threshold=0.95):
    """
    W_learned: (n_atoms, patch_dim)
    W_true: (n_atoms, patch_dim)
    
    for each true atom, finds the best matching learned atom by cosine similarity
    returns fraction of true atoms recovered above threshold
    """
    W_l = W_learned / (np.linalg.norm(W_learned, axis=1, keepdims=True) + 1e-8)
    W_t = W_true / (np.linalg.norm(W_true, axis=1, keepdims=True) + 1e-8)
    
    sim = np.abs(W_l @ W_t.T)  # (n_atoms, n_atoms), abs because sign is arbitrary
    best_match = sim.max(axis=0)  # for each true atom, best cosine with any learned atom
    
    recovered = (best_match >= threshold).mean()
    print(f"Mean best cosine similarity: {best_match.mean():.4f}")
    print(f"Fraction recovered (>{threshold}): {recovered:.4f}")
    return best_match, recovered


def _match_atoms(D1: np.ndarray, D2: np.ndarray):
    """
    Match atoms of D1 to atoms of D2 using the Hungarian algorithm
    on cosine distances. Assumes square dictionaries (same n_components).

    D1, D2: shape (n_components, n_features) — sklearn's components_ layout.

    Returns:
        row_ind, col_ind: matched index arrays
        matched_similarities: per-pair cosine similarities
        mean_sim: mean cosine similarity across matched pairs
    """
    # Guard against dead atoms (zero-norm rows produce NaN cosine distances)
    norms_1 = np.linalg.norm(D1, axis=1, keepdims=True)
    norms_2 = np.linalg.norm(D2, axis=1, keepdims=True)
    if np.any(norms_1 == 0) or np.any(norms_2 == 0):
        raise ValueError(
            "One or more atoms have zero norm. "
            "Remove or replace dead atoms before matching."
        )

    # cost = cosine_distances(D1, D2)          # shape (n_components, n_components), values in [0, 2]
    cost = np.abs(cosine_similarity(D1, D2))
    cost = 1 - cost

    row_ind, col_ind = linear_sum_assignment(cost)
    matched_similarities = 1 - cost[row_ind, col_ind]
    mean_sim = float(matched_similarities.mean())
    return row_ind, col_ind, matched_similarities, mean_sim

def get_live(components):
    dead = find_dead_atoms(components).numpy()
    live = np.array([i for i in range(components.shape[0]) if i not in dead])
    return components[live]

def hungarian_match(all_components: list[np.ndarray]):
    """
    Pairwise similarity matching across runs using the Hungarian algorithm.

    Args:
        all_components: list of arrays, each shape (n_components, n_features).
                        All arrays must have the same shape.

    Returns:
        upper: 1-D array of pairwise similarities for all unique pairs
        stability_score: mean of upper
        best_run_idx: index of the run most similar to all others
        pairwise_sims: (n_runs, n_runs) symmetric similarity matrix, diagonal = 1
    """
    n_runs = len(all_components)

    if n_runs < 2:
        raise ValueError("Need at least 2 runs to compute pairwise similarity.")

    shapes = [d.shape for d in all_components]
    if len(set(shapes)) != 1:
        raise ValueError(
            f"All dictionaries must have the same shape. Got: {shapes}"
        )

    pairwise_sims = np.ones((n_runs, n_runs))
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            _, _, _, mean_sim = _match_atoms(all_components[i], all_components[j])
            pairwise_sims[i, j] = mean_sim
            pairwise_sims[j, i] = mean_sim

    upper = pairwise_sims[np.triu_indices(n_runs, k=1)]
    stability_score = float(upper.mean())

    # Exclude self-similarity (diagonal=1) when ranking runs
    np.fill_diagonal(pairwise_sims, 0)
    mean_sim_per_run = pairwise_sims.sum(axis=1) / (n_runs - 1)
    best_run_idx = int(np.argmax(mean_sim_per_run))
    np.fill_diagonal(pairwise_sims, 1)  # restore diagonal

    return upper, stability_score, best_run_idx, pairwise_sims

def find_dead_atoms(W, threshold=0.1):
    if not isinstance(W, torch.Tensor):
        W = torch.tensor(W)
    
    peak = W.abs().max(dim=1).values
    peak_normalised = peak / peak.max()
    
    return torch.where(peak_normalised < threshold)[0]

In [ ]:
from collections import defaultdict
import torch
import numpy as np
from torch import nn
from torch import optim


def get_alpha(epoch, total_epochs, alpha_start=0.0, alpha_end=1.0):
    # do the last 25% with max_alpha
    epoch_max = int(total_epochs * 0.30)
    return alpha_start + (alpha_end - alpha_start) * (epoch / epoch_max)

class Autoencoder(nn.Module):
    def __init__(self, input_dim, n_components, init_sigma_eps=None):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_components, bias=True),
        )
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        coefficients = self.encoder(x)
        recon = self.decoder(coefficients)
        return recon, coefficients

    def recon_loss(self, x, recons, sigma_eps):
        """Reconstruction loss. MSE"""
        return self.gauss_loss(x, recons) / (sigma_eps * sigma_eps)

    def coefficients_loss(self, codes, sigma_s):
        """L2 loss on encoder"""
        return self.gauss_loss(codes, 0) / (sigma_s * sigma_s)

    def gauss_loss(self, x, mean):
        loss = (x - mean) ** 2
        return torch.sum(loss, 1).mean()

    def weights_loss(self, alpha, sigma_0, W):
        """Vectorized version the Weight loss"""
        W_sq = W**2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
        comp2 = -torch.log(phi)
        return comp1, comp2

def train(
    X,
    n_components,
    alpha=5000,
    sigma_eps=0.1,
    sigma_s=1,
    sigma_0=1,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    weights_algo="cycle",
    verbose=True,
    svd_init=False,
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    sigma_eps: std of noise in the data after modelling the data as a W@S
    sigma_0: std of W, useful to keep very near 0
    sigma_s: sigma for the gaussian distribution for sampling the encoder weights. It indirectly restricts its outputs (the coefficients) to be gaussian
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)

    grads = defaultdict(list)
    losses = defaultdict(list)

    if svd_init:
        print("using svd init for decoder")
        U, s, Vt = np.linalg.svd(X, full_matrices=False)
        model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)

            recon_loss = model.recon_loss(batch, recon, sigma_eps)
            coefficients_loss = model.coefficients_loss(codes, sigma_s)
            comp1, comp2 = model.weights_loss(alpha, sigma_0, model.decoder.weight)
            comp1, comp2 = comp1.sum(), comp2.sum()
            weight_loss = comp1 + comp2

            losses["comp1"].append(comp1.item())
            losses["comp2"].append(comp2.item())
            losses["recon"].append(recon_loss.item())
            losses["codes"].append(coefficients_loss.item())


            loss = recon_loss + weight_loss + coefficients_loss

            optimizer.zero_grad()
            _push_grads(grads, model, comp1, comp2, recon_loss, coefficients_loss)
            loss.backward()
            optimizer.step()

        if verbose and epoch % 200 == 0:
            print(
                f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} weight_loss {weight_loss.sum():.4f} coefficients_loss {coefficients_loss:.4f}"
            )

    with torch.no_grad():
        recon, codes = model(X_t)
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),
        recon.numpy(),
        grads,
        losses,
    )


def train_baseline(
    X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1, verbose=True
):
    """Train the autoencoder with just reconstruction loss, to find an arbitrary linear model which fits the data

    The main training code requires sigma_eps
    the standard deviation of expected gaussian noise
    when the curve is fitted using Y=WX
    We can generally do a simple sweep of hyperparams
    or use simple heuristics
    If the data is linearly "fittable",
    then we get a good starting point
    using this function. 
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            loss = recon_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose and epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

def _push_grads(grads_dict, model, comp1, comp2, recon_loss, coefficients_loss):
    grads = torch.autograd.grad(comp1, [model.decoder.weight], retain_graph=True)
    grads_dict["comp1_decoder"].append(grads[0].mean())

    grads = torch.autograd.grad(comp2, [model.decoder.weight], retain_graph=True)
    grads_dict["comp2_decoder"].append(grads[0].mean())

    grads = torch.autograd.grad(recon_loss, [model.decoder.weight], retain_graph=True)
    grads_dict["recon_decodet"].append(grads[0].mean())

    grads = torch.autograd.grad(recon_loss, [model.encoder[0].weight], retain_graph=True)
    grads_dict["recon_encoder"].append(grads[0].mean())

    grads = torch.autograd.grad(coefficients_loss, [model.encoder[0].weight], retain_graph=True)
    grads_dict["codes_encoder"].append(grads[0].mean())

In [ ]:
# | code-fold: true


X, W_true, codes_true, dim_partition = generate_synthetic_patches(
    patch_dim=9, n_components=3, k=3
)
ws_to_show = [w.reshape(3, 3) for w in W_true]
S(
    ws_to_show,
    (8, 2),
    ncols=3,
    mode=MODE,
    suptitle="ground truth basis vectors \n[bright green = high positive, bright red = high negative, white = near zero]\nA vector is of size 1x9, but is shown as 3x3 for easier visibility",
)
plt.show()
S(
    [X[0].reshape(3, 3)] + ws_to_show,
    (8, 2),
    4,
    suptitle="First input, with its basis components, the coefficient of each component is it's title",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[0][0]:.3f}",
        f"{codes_true[0][1]:.3f}",
        f"{codes_true[0][2]:.3f}",
    ],
)
plt.show()
S(
    [X[1].reshape(3, 3)] + ws_to_show,
    (8, 2),
    4,
    suptitle="Second input",
    mode=MODE,
    ax_titles=[
        "X",
        f"{codes_true[1][0]:.3f}",
        f"{codes_true[1][1]:.3f}",
        f"{codes_true[1][2]:.3f}",
    ],
)
plt.show()

In [ ]:
sigma_eps = 1e-2
sigma_0 = sigma_eps * 5
sigma_s = sigma_0 * 5

alpha = 5000 / (sigma_0 * sigma_0)
run = train(X, 6, 5000, sigma_eps, sigma_s, sigma_0, 1e-3, 1000, 256, "no-cycle", True, False)


In [ ]:

def show_run_plots(grads, losses, title):
    print("TITLE", title)
    _, axes = plt.subplots(1, 3, figsize=(15,3), sharey=True)
    print("grads scaled")
    axes[0].plot(grads["comp1_decoder"], color="red")
    axes[1].plot(grads["comp2_decoder"], color="red")
    axes[2].plot(grads["recon_decodet"], color="green")
    plt.show()

    print("grads unscaled")
    _, axes = plt.subplots(1, 3, figsize=(15,3), sharey=False)
    axes[0].plot(grads["comp1_decoder"], color="red")
    axes[1].plot(grads["comp2_decoder"], color="red")
    axes[2].plot(grads["recon_decodet"], color="green")
    plt.show()

    print("losses")
    _, axes = plt.subplots(1, 4, figsize=(20, 3))
    axes[0].plot(losses["comp1"])
    axes[1].plot(losses["comp2"])
    axes[2].plot(losses["recon"])
    axes[3].plot(losses["codes"])
    plt.show()

In [ ]:
grads = torch.autograd.grad(comp1, [model.decoder.weight], retain_graph=True)
print("comp1 to decodeR", grads[0].std(), grads[0].mean())

grads = torch.autograd.grad(comp2, [model.decoder.weight], retain_graph=True)
print("comp2 to decoder", grads[0].std(), grads[0].mean())

grads = torch.autograd.grad(recon_loss, [model.decoder.weight], retain_graph=True)
print("recon to decoder", grads[0].std(), grads[0].mean())

grads = torch.autograd.grad(recon_loss, [model.encoder[0].weight], retain_graph=True)
print("recon to encoder", grads[0].std(), grads[0].mean())

In [ ]:
from itertools import product

sigma_eps_values = [1e-7, 1e-3, 1e-2]
sigma_0_values = [0.2, 1, 5]
sigma_s_values = [0.2, 1, 5]

combinations = list(product(sigma_eps_values, sigma_0_values, sigma_s_values))

In [ ]:

def do_run(sigma_eps, sigma_0_factor, sigma_s_factor):
    sigma_0 = sigma_eps * sigma_0_factor
    sigma_s = sigma_0 * sigma_s_factor

    alpha = 5000 / (sigma_0 * sigma_0)
    run = train(X, 6, alpha, sigma_eps, sigma_s, sigma_0, 1e-3, 1000, 256, "no-cycle", True, False)
    return run


In [ ]:
import gc

grads, losses = {}, {}
for i, (sigma_eps, sigma_0, sigma_s) in enumerate(combinations):
    print(f"############# {i}/{len(combinations)} #################")
    run = do_run(sigma_eps, sigma_0, sigma_s)
    grads[(sigma_eps, sigma_0, sigma_s)] = run[-2]
    losses[(sigma_eps, sigma_0, sigma_s)] = run[-1]
    gc.collect()

In [ ]:
grads.keys()

In [ ]:
show_run_plots(grads[(1e-03, 0.2, 0.2)], losses[(1e-03, 0.2, 0.2)], str((1e-07, 0.2, 0.2)))

In [ ]:
t = (1e-3, 5, 5)
show_run_plots(grads[t], losses[t], str(t))

In [ ]:
run = do_run(*t)

In [ ]:
model, codes, components, recon, _, _ = run

In [ ]:
S([c.reshape(3,3) for c in components], (8,2), len(components), mode=MODE)
S([c.reshape(3,3) for c in W_true], (8,2), len(components), mode=MODE)
plt.show()

In [ ]:
# recons did not happen at all.  why?
S([recon[0].reshape(3,3), X[0].reshape(3,3)], viztype="local")

In [ ]:
model.decoder.weight.std()

In [ ]:
import gc
gc.collect()

# with empirical stuff

In [ ]:
def init_autoencoder(model, sigma_x, input_dim, n_components, 
                     eps_ratio=100, w_to_eps_ratio=5, alpha_constant=5000.0):
    """
    sigma_x      : std of your data
    eps_ratio    : sigma_x / sigma_eps (default 100)
    w_to_eps_ratio: sigma_0 / sigma_eps (default 5)
    alpha_constant: the c in alpha = c / sigma_0^2
    """
    sigma_eps = sigma_x / eps_ratio
    sigma_0   = sigma_eps * w_to_eps_ratio
    sigma_s   = sigma_x / (np.sqrt(n_components) * sigma_0)   # from K*sigma_0^2*sigma_s^2 = sigma_x^2
    sigma_enc = sigma_s / (np.sqrt(input_dim) * sigma_x)      # from D*sigma_enc^2*sigma_x^2 = sigma_s^2
    alpha     = alpha_constant / (sigma_0 ** 2)

    # decoder = W, init with sigma_0
    # nn.init.normal_(model.decoder.weight, mean=0.0, std=sigma_0)

    # # encoder weights
    # nn.init.normal_(model.encoder[0].weight, mean=0.0, std=sigma_enc)
    # nn.init.zeros_(model.encoder[0].bias)

    return dict(sigma_eps=sigma_eps, sigma_0=sigma_0, sigma_s=sigma_s, 
                sigma_enc=sigma_enc, alpha=alpha)

In [ ]:
from collections import defaultdict
import torch
import numpy as np
from torch import nn
from torch import optim


def get_alpha(epoch, total_epochs, alpha_start=0.0, alpha_end=1.0):
    # do the last 25% with max_alpha
    epoch_max = int(total_epochs * 0.30)
    return alpha_start + (alpha_end - alpha_start) * (epoch / epoch_max)

class Autoencoder(nn.Module):
    def __init__(self, input_dim, n_components, init_sigma_eps=None):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_components, bias=True),
        )
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        coefficients = self.encoder(x)
        recon = self.decoder(coefficients)
        return recon, coefficients

    def recon_loss(self, x, recons, sigma_eps):
        """Reconstruction loss. MSE"""
        return self.gauss_loss(x, recons) / (sigma_eps * sigma_eps)

    def coefficients_loss(self, codes, sigma_s):
        """L2 loss on encoder"""
        return self.gauss_loss(codes, 0) / (sigma_s * sigma_s)

    def gauss_loss(self, x, mean):
        loss = (x - mean) ** 2
        return torch.sum(loss, 1).mean()

    def weights_loss(self, alpha, sigma_0, W):
        """Vectorized version the Weight loss"""
        W_sq = W**2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
        comp2 = -torch.log(phi)
        return comp1, comp2

def train(
    X,
    n_components,
    # alpha=5000,
    # sigma_eps=0.1,
    # sigma_s=1,
    # sigma_0=1,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    # weights_algo="cycle",
    verbose=True,
    svd_init=False,
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)
    p = init_autoencoder(model, X_t.std(), input_dim, n_components, eps_ratio=100, w_to_eps_ratio=5, alpha_constant=5000)
    sigma_eps, sigma_0, sigma_s, _, alpha = p["sigma_eps"], p["sigma_0"], p["sigma_s"], p["sigma_enc"], p["alpha"]
    # return dict(sigma_eps=sigma_eps, sigma_0=sigma_0, sigma_s=sigma_s, 
    #             sigma_enc=sigma_enc, alpha=alpha)

    grads = defaultdict(list)
    losses = defaultdict(list)

    # if svd_init:
    #     print("using svd init for decoder")
    #     U, s, Vt = np.linalg.svd(X, full_matrices=False)
    #     model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)

            recon_loss = model.recon_loss(batch, recon, sigma_eps)
            coefficients_loss = model.coefficients_loss(codes, sigma_s)
            comp1, comp2 = model.weights_loss(alpha, sigma_0, model.decoder.weight)
            comp1, comp2 = comp1.sum(), comp2.sum()
            weight_loss = comp1 + comp2

            losses["comp1"].append(comp1.item())
            losses["comp2"].append(comp2.item())
            losses["recon"].append(recon_loss.item())
            losses["codes"].append(coefficients_loss.item())


            loss = recon_loss + weight_loss + coefficients_loss

            optimizer.zero_grad()
            _push_grads(grads, model, comp1, comp2, recon_loss, coefficients_loss)
            loss.backward()
            optimizer.step()

        if verbose and epoch % 200 == 0:
            print(
                f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} weight_loss {weight_loss.sum():.4f} coefficients_loss {coefficients_loss:.4f}"
            )

    with torch.no_grad():
        recon, codes = model(X_t)
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),
        recon.numpy(),
        grads,
        losses,
    )


def train_baseline(
    X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1, verbose=True
):
    """Train the autoencoder with just reconstruction loss, to find an arbitrary linear model which fits the data

    The main training code requires sigma_eps
    the standard deviation of expected gaussian noise
    when the curve is fitted using Y=WX
    We can generally do a simple sweep of hyperparams
    or use simple heuristics
    If the data is linearly "fittable",
    then we get a good starting point
    using this function. 
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            loss = recon_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose and epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

def _push_grads(grads_dict, model, comp1, comp2, recon_loss, coefficients_loss):
    grads = torch.autograd.grad(comp1, [model.decoder.weight], retain_graph=True)
    grads_dict["comp1_decoder"].append(grads[0].mean())

    grads = torch.autograd.grad(comp2, [model.decoder.weight], retain_graph=True)
    grads_dict["comp2_decoder"].append(grads[0].mean())

    grads = torch.autograd.grad(recon_loss, [model.decoder.weight], retain_graph=True)
    grads_dict["recon_decodet"].append(grads[0].mean())

    grads = torch.autograd.grad(recon_loss, [model.encoder[0].weight], retain_graph=True)
    grads_dict["recon_encoder"].append(grads[0].mean())

    grads = torch.autograd.grad(coefficients_loss, [model.encoder[0].weight], retain_graph=True)
    grads_dict["codes_encoder"].append(grads[0].mean())

In [ ]:
run = train(X, 3, 1e-3, 3000)

In [ ]:
# recons did not happen at all.  why?

S([recon[0].reshape(3,3), X[0].reshape(3,3)], viztype="local", mode=MODE)

In [ ]:
model, codes, components, recon, grads, losses = run

In [ ]:
show_run_plots(grads, losses, "")

In [ ]:
p = {'sigma_eps': (0.0058), 'sigma_0': (0.0292), 'sigma_s': (11.5470), 'sigma_enc': (6.6001), 'alpha': (5880773.5000)}



In [ ]:
print("sigma_0", model.decoder.weight.std(), p["sigma_0"])
print("sigma_enc", model.encoder[0].weight.std(), p["sigma_enc"])
print("codes", codes.std(), p["sigma_s"])

In [ ]:
S([recon[0].reshape(3,3), X[0].reshape(3,3)], viztype="local", mode=MODE)

In [ ]:
S([c.reshape(3,3) for c in components], (8,2), len(components), mode=MODE)
S([c.reshape(3,3) for c in W_true], (8,2), len(components), mode=MODE)
plt.show()

# on olivetti now

In [ ]:
n_row, n_col = 2,5
n_components = n_row * n_col
image_shape = (64, 64)

In [ ]:
import logging

import matplotlib.pyplot as plt
from numpy.random import RandomState

from sklearn import cluster, decomposition
from sklearn.datasets import fetch_olivetti_faces

rng = RandomState(0)

# Display progress logs on stdout
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

faces, _ = fetch_olivetti_faces(return_X_y=True, shuffle=True, random_state=rng)
n_samples, n_features = faces.shape

# Global centering (focus on one feature, centering all samples)
faces_centered = faces - faces.mean(axis=0)

# Local centering (focus on one sample, centering all features)
faces_centered -= faces_centered.mean(axis=1).reshape(n_samples, -1)

print("Dataset consists of %d faces" % n_samples)

def plot_gallery(title, images, n_col=n_col, n_row=n_row, cmap=plt.cm.gray):
    fig, axs = plt.subplots(
        nrows=n_row,
        ncols=n_col,
        figsize=(2.0 * n_col, 2.3 * n_row),
        facecolor="white",
        constrained_layout=True,
    )
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.02, hspace=0, wspace=0)
    fig.set_edgecolor("black")
    fig.suptitle(title, size=16)
    for ax, vec in zip(axs.flat, images):
        vmax = max(vec.max(), -vec.min())
        im = ax.imshow(
            vec.reshape(image_shape),
            cmap=cmap,
            interpolation="nearest",
            vmin=-vmax,
            vmax=vmax,
        )
        ax.axis("off")

    fig.colorbar(im, ax=axs, orientation="horizontal", shrink=0.99, aspect=40, pad=0.01)
    plt.show()

plot_gallery("Faces from dataset", faces_centered[:n_components])

In [ ]:
faces_centered.std()

run = train(faces_centered, n_components, 1e-3, 3000)

In [ ]:
model, codes, components, recon, grads, losses = run

In [ ]:
show_run_plots(grads, losses, "")

In [ ]:
show_gram(components.T)

In [ ]:
plot_gallery("", components)

In [ ]:
S([c.reshape(image_shape) for c in components], (10,5), ncols=5, viztype="gray")

In [ ]:
S([recon[0].reshape(image_shape), faces_centered[0].reshape(image_shape)], viztype="gray")

In [ ]:
codes[0]

In [ ]:
from torch import tensor
p = {'sigma_eps': tensor(0.0012), 'sigma_0': tensor(0.0062), 'sigma_s': tensor(6.3246), 'sigma_enc': tensor(0.8029), 'alpha': tensor(1.3201e+08)}

In [ ]:
print("sigma_0", model.decoder.weight.std(), p["sigma_0"])
print("sigma_enc", model.encoder[0].weight.std(), p["sigma_enc"])
print("codes", codes.std(), p["sigma_s"])

In [ ]:
# use curriculum instead of disjoint
# first fit a baseline then see
from collections import defaultdict
import torch
import numpy as np
from torch import nn
from torch import optim


def get_alpha(epoch, total_epochs, alpha_start=0.0, alpha_end=1.0):
    # do the last 25% with max_alpha
    epoch_max = int(total_epochs * 0.30)
    return alpha_start + (alpha_end - alpha_start) * (epoch / epoch_max)

class Autoencoder(nn.Module):
    def __init__(self, input_dim, n_components, init_sigma_eps=None):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_components, bias=True),
        )
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        coefficients = self.encoder(x)
        recon = self.decoder(coefficients)
        return recon, coefficients

    def recon_loss(self, x, recons, sigma_eps):
        """Reconstruction loss. MSE"""
        return self.gauss_loss(x, recons) / (sigma_eps * sigma_eps)

    def coefficients_loss(self, codes, sigma_s):
        """L2 loss on encoder"""
        return self.gauss_loss(codes, 0) / (sigma_s * sigma_s)

    def gauss_loss(self, x, mean):
        loss = (x - mean) ** 2
        return torch.sum(loss, 1).mean()

    def weights_loss(self, alpha, sigma_0, W):
        """Vectorized version the Weight loss"""
        W_sq = W**2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
        comp2 = -torch.log(phi)
        return comp1, comp2

def train(
    X,
    n_components,
    # alpha=5000,
    # sigma_eps=0.1,
    # sigma_s=1,
    # sigma_0=1,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    # weights_algo="cycle",
    verbose=True,
    svd_init=False,
    initialised_model=None,
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    if initialised_model is None:
        model = Autoencoder(input_dim, n_components)
    else:
        model = initialised_model

    p = init_autoencoder(model, X_t.std(), input_dim, n_components, eps_ratio=100, w_to_eps_ratio=5, alpha_constant=5000)
    sigma_eps, sigma_0, sigma_s, _, alpha = p["sigma_eps"], p["sigma_0"], p["sigma_s"], p["sigma_enc"], p["alpha"]
    print("using", p)

    grads = defaultdict(list)
    losses = defaultdict(list)

    # if svd_init:
    #     print("using svd init for decoder")
    #     U, s, Vt = np.linalg.svd(X, full_matrices=False)
    #     model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)

            recon_loss = model.recon_loss(batch, recon, sigma_eps)
            coefficients_loss = model.coefficients_loss(codes, sigma_s)
            comp1, comp2 = model.weights_loss(alpha, sigma_0, model.decoder.weight)
            comp1, comp2 = comp1.sum(), comp2.sum()
            # weight_loss = comp1 + comp2
            weight_loss = comp1

            losses["comp1"].append(comp1.item())
            losses["comp2"].append(comp2.item())
            losses["recon"].append(recon_loss.item())
            losses["codes"].append(coefficients_loss.item())


            loss = recon_loss + weight_loss + coefficients_loss

            optimizer.zero_grad()
            _push_grads(grads, model, comp1, comp2, recon_loss, coefficients_loss)
            loss.backward()
            optimizer.step()

        if verbose and epoch % 200 == 0:
            print(
                f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} weight_loss {weight_loss.sum():.4f} coefficients_loss {coefficients_loss:.4f}"
            )

    with torch.no_grad():
        recon, codes = model(X_t)
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),
        recon.numpy(),
        grads,
        losses,
    )


def train_baseline(
    X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1, verbose=True
):
    """Train the autoencoder with just reconstruction loss, to find an arbitrary linear model which fits the data

    The main training code requires sigma_eps
    the standard deviation of expected gaussian noise
    when the curve is fitted using Y=WX
    We can generally do a simple sweep of hyperparams
    or use simple heuristics
    If the data is linearly "fittable",
    then we get a good starting point
    using this function. 
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            l2_s = model.gauss_loss(codes, 0)
            l2_w = model.gauss_loss(model.decoder.weight, 0)
            loss = recon_loss + l2_s + l2_w
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose and epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

def _push_grads(grads_dict, model, comp1, comp2, recon_loss, coefficients_loss):
    grads = torch.autograd.grad(comp1, [model.decoder.weight], retain_graph=True)
    grads_dict["comp1_decoder"].append(grads[0].mean())

    grads = torch.autograd.grad(comp2, [model.decoder.weight], retain_graph=True)
    grads_dict["comp2_decoder"].append(grads[0].mean())

    grads = torch.autograd.grad(recon_loss, [model.decoder.weight], retain_graph=True)
    grads_dict["recon_decodet"].append(grads[0].mean())

    grads = torch.autograd.grad(recon_loss, [model.encoder[0].weight], retain_graph=True)
    grads_dict["recon_encoder"].append(grads[0].mean())

    grads = torch.autograd.grad(coefficients_loss, [model.encoder[0].weight], retain_graph=True)
    grads_dict["codes_encoder"].append(grads[0].mean())

In [ ]:
import gc
gc.collect()

In [ ]:
baseline_model, codes, components, recon = train_baseline(faces_centered, n_components, 1e-3, 800)

In [ ]:
baseline_model, codes, components, recon = train_baseline(faces_centered, n_components, 1e-3, 800)
run = train(faces_centered, n_components, 1e-3, 3000, initialised_model=model)

In [ ]:
run = train(faces_centered, n_components, 1e-3, 3000, initialised_model=baseline_model)

In [ ]:
model, codes, components, recon, grads, losses = run

In [ ]:
show_run_plots(grads, losses, "without log term")

In [ ]:
show_run_plots(grads, losses, "")

In [ ]:
show_gram(components.T)

In [ ]:
show_gram(components.T)

In [ ]:
# components with log term
S([c.reshape(image_shape) for c in components], (10,5), 5)
plt.show()

In [ ]:
# components without log term
S([c.reshape(image_shape) for c in components], (10,5), 5)
plt.show()

In [ ]:
plot_gallery("", components)

In [ ]:
# on synthetic data

S([X[0].reshape(3,3)], mode=MODE)


In [ ]:
baseline_model, codes, components, recon = train_baseline(X, 3, 1e-3, 800)
run = train(X, 3, 1e-3, 2000, initialised_model=baseline_model)

In [ ]:
model, codes, components, recon, grads, losses = run

In [ ]:
show_gram(components.T)

In [ ]:
S([c.reshape(3,3) for c in W_true], (8,2), len(W_true), mode=MODE)
plt.show()

In [ ]:
S([c.reshape(3,3) for c in components], (8,2), len(W_true), mode=MODE)
plt.show()

# Finding scales of gradients


# wrt W

## MSE Gradient Scale

$$
\frac{\partial MSE}{\partial w_{ij}} = - 2 s_i \left( \sum_{c=0}^{C} x_c - \sum_{r=0}^R \sum_{c=0}^C s_r w_{rc} \right)
$$

In the sums, cross terms vanish with expectation since the variables are independent.  ($E[s_r s_r'] = E[s_r]E[s_r'] = 0$)

$$
\begin{align*}
\left(\frac{\partial MSE}{\partial W_{ij}}\right)^2 &= 
4 \left[ 
    \left( \sum_{c=0}^C x_c s_i^2 \right)^2 
    + \left( \sum_{r=0}^R \sum_{c=0}^C s_r w_{rc} s_i^2 \right)^2 
    - 2 \left(\sum_{c=0}^C x_c \right) \left( \sum_{r=0}^R \sum_{c=0}^C s_r w_{rc} s_i^2 \right)
\right] \\


&= 
4 \left[ 
    \sum_{c=0}^C E[x_c ^2] E[s_i^2]
    + \sum_{r'}\sum_{c'}\sum_{r}\sum_{c} E[s_r s_r' s_i^2] E[w_{rc} w_{r'c'}]
\right] \\

&= 
4 \left[ 
    \sum_{c=0}^C E[x_c ^2] E[s_i^2]
    + \sum_{r}\sum_{c} E[s_r^2 s_i^2] E[w_{rc}^2]
\right] \\

&= 
4 \left[ 
    C (E[x_c ^2] E[s_i^2])
    + E[w_{rc}^2] \left(\sum_{r \ne i}\sum_{c} E[s_r^2] E[s_i^2] + \sum_c E[s_i^4] \right)
\right] \\


&= 
4 \left[ 
    C (\sigma_x^2 \beta_s^2)
    + \beta_w^2 \left(C(R-1)\beta_s^4 + 3C\beta_s^4 \right)
\right] \\

&= 
4C\beta_s^2 \left[ 
    \sigma_x^2
    + \beta_w^2 \beta_s^2 (R+2)
\right] \\
\end{align*}
$$

---

$$
\left(\frac{\partial MSE}{\partial W_{ij}}\right)^2 = 
4C\beta_s^2 \left[ 
    \sigma_x^2
    + \beta_w^2 \beta_s^2 (R+2)
\right] \\
$$

## Disjoint loss gradient
$$
\frac{\partial \phi(r-1, c)}{\partial w_{ij}} = \begin{cases}
0 & \text{if } j \ne c \\
0 & \text{if } j = c, r \leq i \\
2 \alpha w_{ij} & \text{if } j=c, r \gt i \\
\end{cases}
$$

### First term

$$
\begin{align*}
\frac{\partial}{\partial w_{ij}} \left( \sum_c^C \sum_r^R -\ln(\phi(r-1, c)) \right) &= \sum_c^C \sum_r^R \left( \frac{-\phi'(r-1,c)}{\phi(r-1, c)} \right) \\
&= \sum_{r \gt i}^R \frac{-2\alpha w_{ij}}{\phi(r-1,j)}
\end{align*}
$$

### Second term

$$
\begin{align*}
\frac{\partial}{\partial w_{ij}} \left[ \sum_c^C \sum_r^R \frac{w_{rc}^2 \phi(r-1,c)}{\sigma_0^2}  \right]

&= \frac{1}{\sigma_0^2} 
\left[ 
    \sum_c^C \sum_r^R \phi(r-1,c) \frac{\partial}{\partial w_{ij}} w_{rc}^2 
    + \sum_c^C \sum_r^R  w_{rc}^2 \phi'(r-1, c)
\right] \\

&= \frac{1}{\sigma_0^2} 
\left[ 
    2 w_{ij} \phi(i-1,j)
    + 2 w_{ij} \sum_{r \gt i}  \alpha w_{rj}^2 
\right] \\

&= \frac{2w_{ij}}{\sigma_0^2} 
\left[ 
    1 + \alpha \sum_{r \lt i} w_{rj}^2
    + \alpha \sum_{r \gt i}  w_{rj}^2 
\right] \\

&= \frac{2w_{ij}}{\sigma_0^2} 
\left[ 
    1 + \alpha \sum_{r \ne i} w_{rj}^2
\right] \\
\end{align*}
$$

### final

$$
\frac{\partial}{\partial w_{ij}} DisjointLoss = 
\sum_{r \gt i}^R \frac{-2\alpha w_{ij}}{\phi(r-1,j)} + 
\frac{2w_{ij}}{\sigma_0^2} 
\left[ 
    1 + \alpha \sum_{r \ne i} w_{rj}^2
\right]
$$